In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

import time

from bs4 import BeautifulSoup
import requests

from datetime import datetime

import pandas as pd

In [3]:
url = 'https://www.game.guide/fisch-value-list'

brave_browser = r'/Applications/Brave Browser.app'

options = Options()
options.binary_location = brave_browser
# options.add_argument('--headless')

driver = webdriver.Chrome(options = options)
driver.get(url)

# Scroll first to load all data behind lazy loading
current_height = driver.execute_script("return document.body.scrollHeight")
# print(current_height)

timer = 0.5
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(timer)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == current_height:
        break
    current_height = new_height

# print(current_height)

In [4]:
page = driver.page_source

# Identifying the table
soup = BeautifulSoup(page, "html.parser")
table = soup.find('div', class_='cos-units-grid-5col')
table_row = table.find_all('a')

# Preparing the dataframe
table_title = ["Name", "Category", "Demand", "Volatility", "Rarity", "TrueVal", "Trade Hub", "Proto", "Date"]
df = pd.DataFrame(columns = table_title)

# Preparing mapping to identify item category
category_background = {
    "background: linear-gradient(rgba(153, 129, 87, 0.5) 0%, rgba(153, 129, 87, 0.25) 40%, rgb(17, 17, 17) 100%);": "Boats",
    "background: linear-gradient(rgba(59, 130, 246, 0.5) 0%, rgba(59, 130, 246, 0.25) 40%, rgb(17, 17, 17) 100%);": "Submarines",
    "background: linear-gradient(rgba(255, 0, 0, 0.5) 0%, rgba(255, 0, 0, 0.25) 40%, rgb(17, 17, 17) 100%);": "Rod Skins",
    "background: linear-gradient(rgba(75, 216, 255, 0.5) 0%, rgba(75, 216, 255, 0.25) 40%, rgb(17, 17, 17) 100%);": "Bobbers",
    "background: linear-gradient(rgba(225, 125, 255, 0.5) 0%, rgba(225, 125, 255, 0.25) 40%, rgb(17, 17, 17) 100%);": "Gliders",
    "background: linear-gradient(rgba(170, 85, 255, 0.5) 0%, rgba(170, 85, 255, 0.25) 40%, rgb(17, 17, 17) 100%);": "Booths",
    "background: linear-gradient(rgba(249, 157, 3, 0.5) 0%, rgba(249, 157, 3, 0.25) 40%, rgb(17, 17, 17) 100%);": "Lanterns",
    "background: linear-gradient(rgba(255, 209, 102, 0.5) 0%, rgba(255, 209, 102, 0.25) 40%, rgb(17, 17, 17) 100%);": "Halos",
    "background: linear-gradient(rgba(139, 212, 80, 0.5) 0%, rgba(139, 212, 80, 0.25) 40%, rgb(17, 17, 17) 100%);": "Companion Skins",
    "background: linear-gradient(rgba(207, 214, 223, 0.5) 0%, rgba(207, 214, 223, 0.25) 40%, rgb(17, 17, 17) 100%);": "Rod Sets"
}

# Preparing mapping to identify Demand, Volatility, Rarity
dvr_mapping = {
    "Demand": ["Very Low", "Low", "Medium", "High", "Very High"],
    "Volatility": ["Stable", "Hoarded", "Manipulated", "Rising"],
    "Rarity": ["Common", "Uncommon", "Rare", "Very Rare", "Extremely Rare", "Extinct Wild"]
}

value_to_attribute = {
    value: attribute
    for attribute, values in dvr_mapping.items()
    for value in values
}


In [5]:
# Actual data scrape process

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
convert_dict = {'M': 1000000, 'K': 1000}
df = df.iloc[:0]

templist_catch_new_attributes = []

for data_row in table_row:
    individual_row_data_proper = []

    # Get name
    individual_row_name = data_row.find('div', class_='cos-unit-card-name').get_text().strip()
    individual_row_data_proper.append(individual_row_name)

    # Get category (based off background)
    background = data_row.select_one(".cos-unit-card-image-area")["style"]
    category = category_background.get(background)
    individual_row_data_proper.append(category)
    
    # Get demand, volatility, and rarity
    individual_row_dvr_check = data_row.find('div', class_='cos-card-stats-tr')
    individual_row_dvr = individual_row_dvr_check.find_all('span', class_='cos-card-corner-stat')

    item_attributes = {
        "Demand": '',
        "Volatility": '',
        "Rarity": ''
    }

    for dvr in individual_row_dvr:
        value = dvr.get_text()
        attribute = value_to_attribute.get(value)

        if attribute:
            item_attributes[attribute] = value
        else:
            templist_catch_new_attributes.append(value)

    individual_row_data_proper.append(item_attributes['Demand'])
    individual_row_data_proper.append(item_attributes['Volatility'])
    individual_row_data_proper.append(item_attributes['Rarity'])

    # Get TrueVal, Trade Hub, and Proto value
    individual_row_data = [value.get_text().strip() for value in data_row.find_all('span', class_='cos-unit-card-field-value')]

    # converting TrueVal and Trade Hub data to integers
    for temp in individual_row_data[:2]:
        if temp == 'N/A' or temp == 'C/C' or temp == 'O/C':
            individual_row_data_proper.append(None)
        else:
            temp_list = temp.split()
            temp_value = temp_list[1]

            # checking if they are in Ms or Ks or not, then convert to int
            if (ord(temp_value[-1]) >= 48 and ord(temp_value[-1]) <= 57):
                individual_row_data_proper.append(int(temp_value))
            else:
                factor, exp = temp_value[0:-1], temp_value[-1]
                converted_value = int(float(factor) * convert_dict[exp])

                individual_row_data_proper.append(converted_value)

    # converting Proto data to integers
    if individual_row_data[-1] == 'N/A':
        individual_row_data_proper.append(None)
    else:
        individual_row_data_proper.append(float(individual_row_data[-1].replace(',','')))

    # Get date
    individual_row_data_proper.append(now)
    
    print(individual_row_data_proper)

    length = len(df)
    df.loc[length] = individual_row_data_proper

print("Checking any new attributes -> " + str(templist_catch_new_attributes))

['KR ROAR', 'Boats', '', '', '', None, 945000, None, '2026-09-29 20:13:16']
['Caelinfernum', 'Rod Skins', '', '', '', None, 930000, None, '2026-09-29 20:13:16']
['Fishymotion', 'Rod Skins', '', '', '', None, 378000, None, '2026-09-29 20:13:16']
["Cash Cruisin'", 'Gliders', '', '', '', None, 120000, None, '2026-09-29 20:13:16']
['Moneyheist', 'Boats', '', '', '', None, 80000, None, '2026-09-29 20:13:16']
['Centicute', 'Rod Skins', '', '', '', None, 415000, None, '2026-09-29 20:13:16']
['Whalesong Orcarina', 'Rod Skins', '', '', '', None, 120000, None, '2026-09-29 20:13:16']
['Coco Capy', 'Companion Skins', '', '', '', None, 325000, None, '2026-09-29 20:13:16']
['Puff of Heaven', 'Rod Skins', 'Medium', 'Stable', 'Extremely Rare', None, None, 7200.0, '2026-09-29 20:13:16']
["Cthulu's Revenge", 'Rod Skins', 'Medium', 'Stable', 'Extremely Rare', None, 26300000, 5000.0, '2026-09-29 20:13:16']
['Evangeline', 'Boats', 'High', 'Stable', 'Very Rare', 10800000, 10800000, 2250.0, '2026-09-29 20:13

In [6]:
# Saving data as csv file

df_cleaned = df.fillna('')
df_cleaned.to_csv(r'/Users/wais/Desktop/Projects for Github/Fisch Trade Market Analysis/daily data/fisch_gameguide_scrape_20260929.csv', index = False)
driver.quit()